## Binary Logistic Regression

We will now use an output layer, where each neuron sperately represents two classes - 0 and 1. A model with this type of output layer is called binary logistic regression.

We will use signmoid activation instead of softmax and binary cross-entropy rather than categorical cross-entropy for calculating loss.

### Sigmoid Activation function

$\sigma(x) = \frac{1}{1+\exp^{-x}}$

The derivative of sigmoid is $\sigma (1\ -\ \sigma)$.


In [1]:
import numpy as np
from nnfs.datasets import vertical_data, spiral_data
import nnfs

In [2]:
class Activation_sigmoid:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = 1 / (1 + np.exp(-inputs))

    def backward(self, dvalues):
        self.dinputs = dvalues * (1 - self.output) * self.output


### Binary Cross-Entropy Loss

Binary loss is given as 

$L_{i,j} = -y_{i,j} log(\hat y_{i,j}) - (1 - y_{i,j}) log(1 - \hat y_{i,j})$

Since a model can contain multiple binary losses, each of them, unlike in the cross-entropy loss, outputs its own prediction, loss calculated on a single output is going to be a vector of losses containing one value for each output.

$L_i = \frac{1}{J} \sum_j L_{i,j}$

In [3]:
# Example

# An output from a model with 3 output neuron
outputs = np.array([[1, 3, 4],
                    [2, 4, 6],
                    [0, 5, 10],
                    [5, 10, 15]])

print(np.mean(outputs, axis = -1))

[ 2.66666667  4.          5.         10.        ]


### Binary Cross-Entropy Loss Derivative

The partial derivative will come out to be 

$-\frac{1}{J}(\frac{y_{i,j}}{\hat y_{i,j}} - \frac{1 - y_{i,j}}{1 - \hat y_{i,j}})$

In [4]:
class Loss:
    def regularization_loss(self, layer):

        # 0 by default
        regularization_loss = 0

        # L1 regularization - weights
        if layer.weight_regularizer_l1 > 0:
            regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

        # L2 regularization - weights
        if layer.weight_regularizer_l2 > 0:
            regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

        # L1 regularization - biases
        if layer.bias_regularizer_l1 > 0:
            regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        # L2 regularization - biases
        if layer.bias_regularizer_l2 > 0:
            regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return regularization_loss

    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_BinaryCrossentropy(Loss):

    def forward(self, y_pred, y_true):
        #clipping on both ends to prevent division by 0
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        sample_losses = -(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))
        sample_losses = np.mean(sample_losses, axis = -1)

        return sample_losses
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        outputs = len(dvalues[0])

        clipped_dvalues = np.clip(dvalues, 1e-7, 1 - 1e-7)

        self.dinputs = -(y_true / clipped_dvalues - (1 - y_true) / (1 - clipped_dvalues)) / outputs

        self.dinputs = self.dinputs / samples

In [5]:
X, y = spiral_data(samples = 100, classes = 2)

y = y.reshape(-1,1)
y

array([[0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
    

In [6]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons, weight_regularizer_l1 = 0, weight_regularizer_l2 = 0, bias_regularizer_l1 = 0, bias_regularizer_l2 = 0):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

        # set regularization penalty
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        #Gradients on parameter
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)

        #Gradients on regularization
        # L1 on weights
        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dL1

        # L2 on weights
        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        # L1 on biases
        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1  

        # L2 on biases
        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases       
        
        self.dinputs = np.dot(dvalues, self.weights.T)

class Dropout_layer:
    def __init__(self, rate):
        self.rate = 1 - rate

    def forward(self, inputs):
        self.inputs = inputs
        
        self.binary_mask = np.random.binomial(1, self.rate, size = inputs.shape) / self.rate

        self.output = inputs * self.binary_mask

    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask


class Activation_Relu:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)

            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            #calculate sample-wise gradient
            #and add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

class Activation_sigmoid:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = 1 / (1 + np.exp(-inputs))

    def backward(self, dvalues):
        self.dinputs = dvalues * (1 - self.output) * self.output

class Loss:
    def regularization_loss(self, layer):

        # 0 by default
        regularization_loss = 0

        # L1 regularization - weights
        if layer.weight_regularizer_l1 > 0:
            regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

        # L2 regularization - weights
        if layer.weight_regularizer_l2 > 0:
            regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

        # L1 regularization - biases
        if layer.bias_regularizer_l1 > 0:
            regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        # L2 regularization - biases
        if layer.bias_regularizer_l2 > 0:
            regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return regularization_loss

    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues) # Number of samples

        #Number of labels in every sample
        #We'll use the first sample to count them
        labels = len(dvalues[0])

        #If labels are sparse then, turn then into one-hot vector
        if len(y_true.shape) == 1:
            y_true =   np.eye(labels)[y_true]

        #calculate gradient
        self.dinputs = -y_true / dvalues

        #Normalize gradient
        self.dinputs = self.dinputs / samples

class Activation_Softmax_Loss_CategoricalCrossentropy():

    def __init__(self):
        self.activation = Activation_softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        #output layer activation function
        self.activation.forward(inputs)

        #set the output
        self.output = self.activation.output

        #calculate and return loss value
        return self.loss.calculate(self.output, y_true)
    
    def backward(self, dvalues, y_true):

        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs/samples

class Loss_BinaryCrossentropy(Loss):

    def forward(self, y_pred, y_true):
        #clipping on both ends to prevent division by 0
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        sample_losses = -(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))
        sample_losses = np.mean(sample_losses, axis = -1)

        return sample_losses
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        outputs = len(dvalues[0])

        clipped_dvalues = np.clip(dvalues, 1e-7, 1 - 1e-7)

        self.dinputs = -(y_true / clipped_dvalues - (1 - y_true) / (1 - clipped_dvalues)) / outputs

        self.dinputs = self.dinputs / samples

In [7]:
class Optimizer_Adagrad:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache += layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


class Optimizer_RMSProp:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7, rho = 0.9):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.rho = rho

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += self.rho * layer.weight_cache + (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache += self.rho * layer.bias_cache + (1 - self.rho) * layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


class Optimizer_Adam:

    def __init__(self, learning_rate = 0.001, decay = 0, epsilon = 1e-7, beta_1 = 0.9, beta_2 = 0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        #Update moementum with current gradients
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        # Update cache with squared current gradients
        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache += self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases ** 2

        #Get corrected cache
        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases += -self.current_learning_rate * bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


In [8]:
X, y = spiral_data(samples = 100, classes = 2)

y = y.reshape(-1,1)
# Dense layer with 2 input and 64 output values (64 neurons).
dense1 = Dense_layer(2, 64, weight_regularizer_l2=5e-4, bias_regularizer_l2=5e-4)

activation1 = Activation_Relu()

#Create second dense layer that take output from previous layer hence 3 inputs and 3 output values
dense2 = Dense_layer(64, 1)

activation2 = Activation_sigmoid()

#We are now using new class to calculate final softmax activation and loss calculation.
loss_activation = Loss_BinaryCrossentropy()

In [9]:
optimizer = Optimizer_Adam(decay = 5e-5, learning_rate=0.05)

for epoch in range (10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    activation2.forward(dense2.output)
    data_loss = loss_activation.calculate(activation2.output, y)

    regularization_loss = loss_activation.regularization_loss(dense1) + loss_activation.regularization_loss(dense2)

    loss = data_loss + regularization_loss

    predictions = (activation2.output > 0.5) * 1
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f} ( ' +
              f'data_loss: {data_loss:.3f}, ' +
              f'reg_loss: {regularization_loss:.3f}), ' +
              f'lr: {optimizer.current_learning_rate}')

    loss_activation.backward(activation2.output, y)
    activation2.backward(loss_activation.dinputs)
    dense2.backward(activation2.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

epoch: 0, acc: 0.480, loss: 0.693 ( data_loss: 0.693, reg_loss: 0.000), lr: 0.05
epoch: 100, acc: 0.640, loss: 0.624 ( data_loss: 0.621, reg_loss: 0.003), lr: 0.04975371909050202
epoch: 200, acc: 0.725, loss: 0.563 ( data_loss: 0.561, reg_loss: 0.003), lr: 0.049507401356502806
epoch: 300, acc: 0.750, loss: 0.537 ( data_loss: 0.534, reg_loss: 0.003), lr: 0.0492635105177595
epoch: 400, acc: 0.750, loss: 0.525 ( data_loss: 0.522, reg_loss: 0.003), lr: 0.04902201088288642
epoch: 500, acc: 0.750, loss: 0.495 ( data_loss: 0.493, reg_loss: 0.003), lr: 0.048782867456949125
epoch: 600, acc: 0.800, loss: 0.472 ( data_loss: 0.470, reg_loss: 0.002), lr: 0.04854604592455945
epoch: 700, acc: 0.810, loss: 0.453 ( data_loss: 0.451, reg_loss: 0.002), lr: 0.048311512633460556
epoch: 800, acc: 0.805, loss: 0.446 ( data_loss: 0.443, reg_loss: 0.002), lr: 0.04807923457858551
epoch: 900, acc: 0.800, loss: 0.441 ( data_loss: 0.439, reg_loss: 0.002), lr: 0.04784917938657352
epoch: 1000, acc: 0.825, loss: 0.42

<ipython-input-7-c25e4f7fb16f>:100: RuntimeWarning: overflow encountered in add
  layer.bias_cache += self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases ** 2
<ipython-input-7-c25e4f7fb16f>:104: RuntimeWarning: overflow encountered in true_divide
  bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))


epoch: 1100, acc: 0.810, loss: 0.413 ( data_loss: 0.410, reg_loss: 0.002), lr: 0.04739561116640599
epoch: 1200, acc: 0.825, loss: 0.396 ( data_loss: 0.394, reg_loss: 0.002), lr: 0.04717203641681212
epoch: 1300, acc: 0.845, loss: 0.388 ( data_loss: 0.386, reg_loss: 0.002), lr: 0.04695056105920466
epoch: 1400, acc: 0.845, loss: 0.380 ( data_loss: 0.377, reg_loss: 0.002), lr: 0.04673115566147951
epoch: 1500, acc: 0.855, loss: 0.369 ( data_loss: 0.366, reg_loss: 0.002), lr: 0.046513791339132055
epoch: 1600, acc: 0.865, loss: 0.360 ( data_loss: 0.358, reg_loss: 0.002), lr: 0.04629843974258068
epoch: 1700, acc: 0.865, loss: 0.360 ( data_loss: 0.358, reg_loss: 0.002), lr: 0.046085073044840774
epoch: 1800, acc: 0.870, loss: 0.352 ( data_loss: 0.350, reg_loss: 0.002), lr: 0.04587366392953806
epoch: 1900, acc: 0.865, loss: 0.345 ( data_loss: 0.342, reg_loss: 0.002), lr: 0.04566418557925019
epoch: 2000, acc: 0.885, loss: 0.318 ( data_loss: 0.316, reg_loss: 0.002), lr: 0.045456611664166556
epoch: 

epoch: 9400, acc: 0.975, loss: 0.113 ( data_loss: 0.107, reg_loss: 0.006), lr: 0.034014762406884586
epoch: 9500, acc: 0.975, loss: 0.112 ( data_loss: 0.106, reg_loss: 0.006), lr: 0.03389945421878708
epoch: 9600, acc: 0.975, loss: 0.112 ( data_loss: 0.106, reg_loss: 0.006), lr: 0.033784925166390756
epoch: 9700, acc: 0.975, loss: 0.111 ( data_loss: 0.106, reg_loss: 0.006), lr: 0.03367116737937304
epoch: 9800, acc: 0.975, loss: 0.111 ( data_loss: 0.105, reg_loss: 0.006), lr: 0.033558173093056816
epoch: 9900, acc: 0.970, loss: 0.111 ( data_loss: 0.105, reg_loss: 0.006), lr: 0.0334459346466437
epoch: 10000, acc: 0.970, loss: 0.110 ( data_loss: 0.104, reg_loss: 0.006), lr: 0.03333444448148271
